# Chapter 29 Lab — Statistical Power and Sample Size (Python)

In this lab you will build an fMRI power calculator from first principles and use it to answer
the questions every study proposal must face: *How many participants do I need? What is the
smallest effect I can detect? And how much will the "winner's curse" inflate my post hoc effect
sizes?* We compute power analytically from the noncentral t distribution, verify it by
simulation, quantify the cost of multiple comparisons correction, and reproduce the core of the
univariate-vs-multivariate power argument from the BWAS debate (Marek et al., 2022).

**How to run this:** This notebook runs in the browser (Pyodide), on
[Colab](https://colab.research.google.com/), or locally — it uses only numpy, scipy, and
matplotlib on simulated data, and every cell runs in seconds. It accompanies the
[Chapter 29 tutorial page](../ch29-statistical-power-and-sample-size.md).

*Calculations mirror the CANlab power utilities and simulation scripts behind the book's
figures (github.com/canlab).*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(29)
plt.rcParams["figure.dpi"] = 90

## Step 1 — Power for a one-sample t-test, analytically

For a one-sample t-test with true effect size $d = \mu/\sigma$ and sample size $N$, the test
statistic follows a **noncentral t distribution** with $N-1$ degrees of freedom and
noncentrality parameter $\delta = d\sqrt{N}$. Power is the probability that this statistic
exceeds the two-tailed critical value:

$$
\text{power} = 1 - F_{nct}\!\left(t_{crit};\; N-1,\; d\sqrt{N}\right)
$$

For a two-sample (balanced) comparison, $\delta = d\sqrt{n/2}$ with $n$ per group and
$df = 2n - 2$ — which is why group comparisons need roughly **4× the total sample**.
For correlations we use the Fisher z approximation: $\text{atanh}(r)$ is approximately normal
with standard error $1/\sqrt{N-3}$.

In [ ]:
def power_one_sample(d, n, alpha=0.05):
    """Power of a two-tailed one-sample t-test with true effect size d."""
    n = np.asarray(n, dtype=float)
    df = n - 1
    t_crit = stats.t.ppf(1 - alpha / 2, df)
    return 1 - stats.nct.cdf(t_crit, df, d * np.sqrt(n))

def power_two_sample(d, n_per_group, alpha=0.05):
    """Power of a two-tailed two-sample t-test, balanced groups of size n_per_group."""
    n = np.asarray(n_per_group, dtype=float)
    df = 2 * n - 2
    t_crit = stats.t.ppf(1 - alpha / 2, df)
    return 1 - stats.nct.cdf(t_crit, df, d * np.sqrt(n / 2))

def power_correlation(r, n, alpha=0.05):
    """Power to detect a correlation r (two-tailed), Fisher z approximation."""
    n = np.asarray(n, dtype=float)
    z_crit = stats.norm.ppf(1 - alpha / 2)
    return stats.norm.cdf(np.sqrt(n - 3) * np.arctanh(r) - z_crit)

def n_for_power(power_fn, effect, alpha=0.05, target=0.80, n_max=5000):
    """Smallest N (or n per group) achieving the target power."""
    n = np.arange(3, n_max)
    pow_curve = power_fn(effect, n, alpha)
    idx = np.argmax(pow_curve >= target)
    return int(n[idx]) if pow_curve[idx] >= target else None

# Quick check against benchmark values
for d in [0.2, 0.5, 0.8]:
    print(f"d = {d}: N = {n_for_power(power_one_sample, d):>4d} "
          f"for 80% power (one-sample, p < .05 two-tailed)")

A "medium" effect of $d = 0.5$ needs about **N = 34** — matching standard power software
(e.g., G\*Power). Now draw the classic power curves: power as a function of $N$ for several
effect sizes.

In [ ]:
d_vals = [0.2, 0.3, 0.4, 0.5, 0.8]
N = np.arange(3, 251)

fig, ax = plt.subplots(figsize=(7, 4.5))
for d in d_vals:
    ax.plot(N, power_one_sample(d, N), label=f"d = {d}")
    n80 = n_for_power(power_one_sample, d)
    if n80 is not None and n80 <= N.max():
        ax.plot([n80, n80], [0, 0.8], ":", color="gray", lw=1)
ax.axhline(0.8, ls="--", color="k", lw=1)
ax.set(xlabel="Sample size (N)", ylabel="Power",
       title="One-sample t-test power, p < .05 two-tailed")
ax.legend(); ax.set_ylim(0, 1.02); plt.tight_layout()

Each curve rises toward 1 as $N$ grows, but the sample size needed for 80% power
(dotted drop lines) explodes as effects shrink: $d = 0.8$ needs ~15 participants, $d = 0.2$
needs ~199. **Sanity check by simulation** — power is just the long-run fraction of
significant results, so we can verify the analytic curve with brute force.

In [ ]:
def simulated_power(d, n, alpha=0.05, n_sims=2000):
    """Fraction of simulated one-sample experiments reaching p < alpha."""
    dat = d + rng.standard_normal((n_sims, n))
    t, p = stats.ttest_1samp(dat, 0.0, axis=1)
    return np.mean(p < alpha)

print(f"{'N':>4} {'analytic':>9} {'simulated':>10}")
for n in [10, 20, 34, 50, 80]:
    print(f"{n:>4} {power_one_sample(0.5, n):>9.3f} {simulated_power(0.5, n):>10.3f}")

Analytic and simulated power agree to within Monte Carlo error. From here on we trust the
analytic formulas.

## Step 2 — The cost of multiple comparisons

A mass-univariate analysis cannot use $\alpha = 0.05$ per voxel. Correction pushes the
effective per-test threshold to roughly:

| Threshold | Typical use |
|---|---|
| $p < 0.05$ | one pre-registered ROI test |
| $p < 0.001$ | uncorrected mapping; often approximates FDR $q < .05$ |
| $p < 0.05/1000$ | Bonferroni over ~1,000 parcels/tests |
| $p < 4.26\times 10^{-6}$ | whole-brain FWER (permutation-based, empirical average) |

How does the required sample size change across these thresholds?

In [ ]:
alphas = {"p < .05 (single ROI)": 0.05,
          "p < .001 (~FDR)": 0.001,
          "Bonferroni, 1000 tests": 0.05 / 1000,
          "FWER whole brain": 4.26e-6}

d_grid = [0.2, 0.3, 0.5, 0.8]
print(f"{'threshold':<24}" + "".join(f"  d={d:<5}" for d in d_grid))
for name, a in alphas.items():
    row = [n_for_power(power_one_sample, d, alpha=a) for d in d_grid]
    print(f"{name:<24}" + "".join(f"  {n:<6}" for n in row))

Reading down each column: correcting for multiplicity multiplies the required sample by
~3–4×. A medium effect ($d = 0.5$) needs **34** participants for one ROI test but about
**120** under whole-brain FWER correction. The same applies to brain–behavior correlations —
reproduce the book's Figure 29.2 numbers:

In [ ]:
r_grid = [0.1, 0.2, 0.3, 0.4, 0.5]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
N = np.arange(4, 2001)
for ax, (name, a) in zip(axes, [("p < .05 two-tailed", 0.05), ("p < .001", 0.001)]):
    for r in r_grid:
        ax.plot(N, power_correlation(r, N, alpha=a), label=f"r = {r}")
    ax.axhline(0.8, ls="--", color="k", lw=1)
    ax.set(xlabel="Participants (N)", title=f"Correlation power, {name}", xlim=(0, 1000))
axes[0].set_ylabel("Power"); axes[0].legend(loc="lower right")
plt.tight_layout()

print("N for 80% power to detect a correlation:")
print(f"{'r':>5} {'p<.05':>8} {'p<.001':>8} {'FWER':>8}")
for r in r_grid:
    ns = [n_for_power(power_correlation, r, alpha=a) for a in (0.05, 0.001, 4.26e-6)]
    print(f"{r:>5} " + " ".join(f"{n or '>5000':>8}" for n in ns))

These match the book's reference values closely (small differences reflect the Fisher z
approximation): detecting $r = 0.5$ needs ~28 participants for a single ROI, ~53 at
$p < .001$, and ~100 with FWER correction — while $r = 0.1$ needs **~780, ~1,540, and
~2,800** respectively. Small correlations and whole-brain search are a brutal combination.

## Step 3 — Minimum detectable effect size

Power analysis can be inverted: given the sample size you can afford, what is the smallest
effect you have an 80% chance of detecting? This *minimum detectable effect size* (MDES) is an
honest summary of a study's sensitivity — and a useful line for grant proposals.

In [ ]:
def min_detectable_d(n, alpha, target=0.80, d_grid=np.arange(0.05, 3.001, 0.005)):
    """Smallest one-sample d detectable with the target power at sample size n."""
    pow_curve = power_one_sample(d_grid, np.full_like(d_grid, n), alpha)
    idx = np.argmax(pow_curve >= target)
    return d_grid[idx] if pow_curve[idx] >= target else np.nan

def min_detectable_r(n, alpha, target=0.80, r_grid=np.arange(0.02, 0.9901, 0.002)):
    """Smallest correlation detectable with the target power at sample size n."""
    pow_curve = power_correlation(r_grid, np.full_like(r_grid, n), alpha)
    idx = np.argmax(pow_curve >= target)
    return r_grid[idx] if pow_curve[idx] >= target else np.nan

n_grid = [30, 50, 100, 200, 500, 1000]
print("Minimum detectable effect with 80% power (one-sample d | correlation r):")
print(f"{'N':>6} {'d, p<.05':>9} {'d, FWER':>9} {'r, p<.05':>10} {'r, FWER':>9}")
for n in n_grid:
    print(f"{n:>6} {min_detectable_d(n, .05):>9.2f} {min_detectable_d(n, 4.26e-6):>9.2f}"
          f" {min_detectable_r(n, .05):>10.2f} {min_detectable_r(n, 4.26e-6):>9.2f}")

In [ ]:
d_grid = np.arange(0.1, 1.21, 0.005)
n_needed = np.array([n_for_power(power_one_sample, d, alpha=4.26e-6) or np.nan
                     for d in d_grid])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(d_grid, n_needed, color="0.25", lw=3)
for n in [30, 50, 100, 200, 500, 1000]:
    mdes = min_detectable_d(n, 4.26e-6)
    ax.plot([mdes, mdes], [0, n], color="orange", lw=2)
    ax.annotate(f"N = {n}\nd = {mdes:.2f}", (mdes, n), textcoords="offset points",
                xytext=(6, 8), fontsize=8)
ax.set(xlabel="Effect size (d)", ylabel="N needed for 80% power",
       title="Whole-brain FWER correction, one-sample test", ylim=(0, 1300))
plt.tight_layout()

With whole-brain FWER correction, a typical N = 30 study is only powered for *very
large* effects ($d \gtrsim 1.2$) — far larger than the $d \approx 0.5$ typical of task effects
in individual voxels. This is the quantitative heart of the "power failure" concern.

## Step 4 — The winner's curse: effect size inflation at a threshold

Now the other side of the coin. When we *estimate* effect sizes only in voxels that survived
thresholding, the estimates are biased upward: voxels are selected partly because their noise
favored the hypothesis. We simulate a brain-full of voxels **all sharing the same true effect**
($d = 0.5$, the true value in the book's Figure 29.1 simulation) and compare post hoc estimates
to the truth.

In [ ]:
n_sub, n_vox, d_true = 30, 20000, 0.5

dat = d_true + rng.standard_normal((n_sub, n_vox))     # subjects x voxels
t, p = stats.ttest_1samp(dat, 0.0)
d_hat = t / np.sqrt(n_sub)                              # observed effect size per voxel

sig = p < 0.001
print(f"True effect size:                 d = {d_true:.2f}")
print(f"Mean estimate, ALL voxels:        d = {d_hat.mean():.2f}   (unbiased)")
print(f"Mean estimate, significant only:  d = {d_hat[sig].mean():.2f}   "
      f"({100 * (d_hat[sig].mean() / d_true - 1):.0f}% inflated)")
print(f"Voxels significant at p < .001:   {sig.mean() * 100:.1f}% "
      f"(= power at this threshold)")

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(-0.2, 1.4, 60)
ax.hist(d_hat, bins=bins, color="0.75", label="all voxels")
ax.hist(d_hat[sig], bins=bins, color="crimson", alpha=0.75,
        label="significant (p < .001)")
ax.axvline(d_true, color="k", ls=":", lw=2, label="true d = 0.5")
ax.set(xlabel="Estimated effect size ($\hat{d}$)", ylabel="Number of voxels",
       title="Winner's curse: selection inflates post hoc effect sizes")
ax.legend(); plt.tight_layout()

Every voxel has the same true effect, yet the significant subset (red) sits almost
entirely to the right of the truth. Two factors govern the inflation — threshold stringency
and sample size:

In [ ]:
thresholds = [0.05, 0.005, 0.001, 4.26e-6]
sample_sizes = [15, 30, 60, 120]

print(f"{'N':>5} " + "".join(f"  p<{a:<9.2g}" for a in thresholds))
for n in sample_sizes:
    dat = d_true + rng.standard_normal((n, n_vox))
    t, p = stats.ttest_1samp(dat, 0.0)
    d_hat = t / np.sqrt(n)
    row = []
    for a in thresholds:
        s = p < a
        row.append(f"{d_hat[s].mean():>11.2f}" if s.sum() >= 10 else f"{'--':>11}")
    print(f"{n:>5} " + " ".join(row))
print(f"\n(True d = {d_true} everywhere. '--' = fewer than 10 significant voxels.)")

Reading across each row, stricter thresholds select luckier noise and inflate the
estimate more; reading down each column, larger samples shrink the bias (with N = 120,
significant voxels barely overestimate). **Paradoxically, correcting for multiple comparisons
makes false positives rarer but post hoc effect sizes more inflated.** This is why effect
sizes for power analysis should come from independent data — an a priori ROI, a pre-defined
pattern, or a replication sample — never from the significant voxels of the same map.

## Step 5 — The BWAS debate: univariate vs. multivariate effects

Marek, Tervo-Clemmens et al. (2022) showed that for brain-wide association studies
(correlating resting-state connectivity or structure with individual differences in behavior),
the largest *univariate* effects are around $r \approx 0.1$. Multivariate models aggregating
signal across the brain achieve up to $r \approx 0.4$ in the same data. How different are the
sample size requirements?

In [ ]:
effects = {"best univariate edge (r = 0.095)": 0.095,
           "median multivariate (r = 0.11)": 0.11,
           "75th pct multivariate (r = 0.18)": 0.18,
           "best multivariate (r = 0.39)": 0.39}

N = np.arange(4, 2501)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for name, r in effects.items():
    ax.plot(N, power_correlation(r, N), label=name)
ax.axhline(0.8, ls="--", color="k", lw=1)
ax.set(xlabel="Sample size (N)", ylabel="Power",
       title="Power for BWAS-scale effects, p < .05 (cf. Marek et al. 2022)")
ax.legend(loc="lower right", fontsize=9); plt.tight_layout()

n_uni = n_for_power(power_correlation, 0.095)
n_multi = n_for_power(power_correlation, 0.39)
print(f"N for 80% power: univariate r=0.095 -> {n_uni};  multivariate r=0.39 -> {n_multi}")
print(f"Sample size reduction: {n_uni / n_multi:.0f}-fold")

The best univariate effects need samples in the **thousands**, while multivariate
effects of $r \approx 0.4$ are detectable with $N$ in the **tens to low hundreds** — roughly a
16-fold reduction, before any multiple comparisons correction (a multivariate model yields
*one* test, so none is needed). Task-evoked multivariate patterns can be far stronger still
($d > 3$ for some validated signatures), detectable in very small samples.

## Wrap-up

- Power depends on the **effect size** in the group analysis — the final common pathway for
  all design, acquisition, and analysis choices — and on the **effective alpha** after
  multiple comparisons correction.
- Required samples grow explosively as effects shrink: $r = 0.5$ needs ~28 participants for
  one test; $r = 0.1$ needs ~780 — and several-fold more under correction. Two-group
  comparisons need ~4× the total sample of one-sample tests.
- The **minimum detectable effect size** tells you what your planned N can honestly claim to
  test.
- Post hoc effect sizes from significant voxels are **inflated by selection** — more so with
  small samples and strict thresholds — so base power analyses on unbiased, independent
  estimates.
- Aggregating signal — a priori ROIs, networks, or **multivariate patterns** — trades
  voxel-level localization for dramatically larger effects, fewer tests, and higher power.